# Keyword & Text Comparison Report

So sanh **keyword** va **evidence text** giua pipeline output va ground truth.

Files: `1.pdf`, `1_original_long.pdf`, `2.pdf`, `4.pdf`, `5.pdf`, `6.pdf`

| Column | Description |
|---|---|
| `Provision (GT)` | Ten provision trong ground truth |
| `GT Keywords` | `specific_keywords` trong ground truth |
| `Output Keywords` | `representative_keyword` + `related_keywords` tu output |
| `KW Match` | check_mark neu >= 1 GT keyword xuat hien trong output keywords (normalized) |
| `GT Text` | `text` trong ground truth |
| `Output Exact Text` | `exact_text` tu evidence output (best match) |
| `Text Match` | check_mark neu token-overlap >= 55% |


In [1]:
from __future__ import annotations
import json, re, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

OUTPUT_DIR       = ROOT / 'data' / 'output'
GROUND_TRUTH_DIR = ROOT / 'data' / 'ground_truth'
TARGET_STEMS     = ['1', '1_original_long', '2', '4', '5', '6']

print('ROOT            :', ROOT)
print('OUTPUT_DIR      :', OUTPUT_DIR)
print('GROUND_TRUTH_DIR:', GROUND_TRUTH_DIR)


ROOT            : d:\keyword_pipeline
OUTPUT_DIR      : d:\keyword_pipeline\data\output
GROUND_TRUTH_DIR: d:\keyword_pipeline\data\ground_truth


In [2]:
def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def norm(text):
    s = str(text or '').strip().lower()
    s = re.sub(r'[\u00a0\u200b\u200c\u200d\ufeff]', ' ', s)
    s = re.sub(r'[\u2010-\u2015\u2212]', '-', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def norm_kw(text):
    s = re.sub(r'[^a-z0-9]+', ' ', norm(text))
    return ' '.join(s.split())

def strip_version(name):
    return re.sub(r'\(\d+\)(?=\.[^.]+$)', '', name)

TEXT_MATCH_THRESHOLD = 0.55

def token_overlap(a, b):
    ta, tb = set(norm(a).split()), set(norm(b).split())
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(len(ta), len(tb))

def kw_match(gt_kws, out_kws):
    out_norm = {norm_kw(k) for k in out_kws}
    hits = [k for k in gt_kws if norm_kw(k) in out_norm]
    return bool(hits), hits

def text_match(gt_text, out_texts):
    if not gt_text or not out_texts:
        return False, 0.0, ''
    best_r, best_c = 0.0, ''
    for c in out_texts:
        r = token_overlap(gt_text, c)
        if r > best_r:
            best_r, best_c = r, c
    return best_r >= TEXT_MATCH_THRESHOLD, best_r, best_c

def trunc(text, n=150):
    t = str(text or '').replace('\n', ' ').replace('\r', '')
    return (t[:n] + '...') if len(t) > n else t

print('Utilities loaded OK')


Utilities loaded OK


In [3]:
def find_output(stem):
    c = OUTPUT_DIR / f'{stem}.json'
    if c.exists():
        return c
    for f in OUTPUT_DIR.glob('*.json'):
        try:
            d = load_json(f)
            dn = strip_version(d.get('document_name', '')).lower()
            if dn == f'{stem.lower()}.pdf':
                return f
        except Exception:
            pass
    return None

def find_gt(stem):
    for f in GROUND_TRUTH_DIR.glob('*.json'):
        try:
            d = load_json(f)
            dn = strip_version(d.get('document', {}).get('document_name', '')).lower()
            if dn == f'{stem.lower()}.pdf':
                return f
        except Exception:
            pass
    return None

pairs = []
for stem in TARGET_STEMS:
    op = find_output(stem)
    gp = find_gt(stem)
    if op and gp:
        status = 'OK'
    elif gp:
        status = 'MISSING_OUTPUT'
    elif op:
        status = 'MISSING_GT'
    else:
        status = 'MISSING_BOTH'
    pairs.append({'stem': stem, 'output': op, 'gt': gp, 'status': status})
    op_name = op.name if op else 'MISSING'
    gp_name = gp.name if gp else 'MISSING'
    print(f'{stem:<25} output={op_name:<40} gt={gp_name}')


1                         output=1.json                                   gt=ground_truth_1.json
1_original_long           output=1_original.json                          gt=ground_truth_1_original_long.json
2                         output=2.json                                   gt=ground_truth_2.json
4                         output=4.json                                   gt=ground_truth_4.json
5                         output=MISSING                                  gt=ground_truth_5.json
6                         output=6.json                                   gt=ground_truth_6.json


In [4]:
def build_out_index(out_data):
    idx = {}
    for g in out_data.get('keyword_groups', []):
        k = norm_kw(g.get('representative_keyword', ''))
        if k and k not in idx:
            idx[k] = g
    return idx

def get_out_kws(g):
    rep = g.get('representative_keyword', '')
    return ([rep] if rep else []) + (g.get('related_keywords') or [])

def get_exact_texts(g):
    return [e['exact_text'] for e in g.get('evidences', []) if e.get('exact_text')]

def compare_pair(pair):
    gt_data  = load_json(pair['gt'])
    out_data = load_json(pair['output'])
    provisions = gt_data.get('document', {}).get('provisions', [])
    out_idx    = build_out_index(out_data)
    rows = []
    for prov in provisions:
        gt_prov  = prov.get('provision', '')
        gt_kws   = prov.get('specific_keywords', [])
        gt_text  = prov.get('text', '')
        # find matching output group
        og = out_idx.get(norm_kw(gt_prov))
        if og is None:
            for kw in gt_kws:
                og = out_idx.get(norm_kw(kw))
                if og:
                    break
        if og is None:
            rows.append({
                'provision': gt_prov, 'gt_kws': gt_kws,
                'out_kws': [], 'kw_ok': False, 'kw_hits': [],
                'gt_text': gt_text, 'out_texts': [],
                'txt_ok': False, 'txt_ratio': 0.0, 'best_txt': '', 'found': False
            })
        else:
            out_kws   = get_out_kws(og)
            out_texts = get_exact_texts(og)
            kw_ok, kw_hits    = kw_match(gt_kws, out_kws)
            txt_ok, ratio, bt = text_match(gt_text, out_texts)
            rows.append({
                'provision': gt_prov, 'gt_kws': gt_kws,
                'out_kws': out_kws, 'kw_ok': kw_ok, 'kw_hits': kw_hits,
                'gt_text': gt_text, 'out_texts': out_texts,
                'txt_ok': txt_ok, 'txt_ratio': ratio, 'best_txt': bt, 'found': True
            })
    return rows

all_results = {}
for pair in pairs:
    all_results[pair['stem']] = compare_pair(pair) if pair['status'] == 'OK' else []
print('Comparison done.')


Comparison done.


In [5]:
from IPython.display import Markdown, display

def pct(a, b): return f'{a/b:.0%}' if b else 'N/A'

lines = ['## Summary', '',
         '| File | GT Provisions | Found in Output | KW Match | Text Match |',
         '|---|:---:|:---:|:---:|:---:|']

for pair in pairs:
    stem  = pair['stem']
    rows  = all_results[stem]
    if pair['status'] == 'MISSING_OUTPUT':
        n = load_json(pair['gt']).get('total_ground_truth_provisions', '?')
        lines.append(f'| **{stem}.pdf** | {n} | output missing | — | — |')
    elif pair['status'] != 'OK':
        lines.append(f'| **{stem}.pdf** | — | {pair["status"]} | — | — |')
    else:
        n       = len(rows)
        found   = sum(1 for r in rows if r['found'])
        kw_pass = sum(1 for r in rows if r['kw_ok'])
        tx_pass = sum(1 for r in rows if r['txt_ok'])
        lines.append(f'| **{stem}.pdf** | {n} | {found} ({pct(found,n)}) '
                     f'| {kw_pass} ({pct(kw_pass,n)}) | {tx_pass} ({pct(tx_pass,n)}) |')

display(Markdown('\n'.join(lines)))


## Summary

| File | GT Provisions | Found in Output | KW Match | Text Match |
|---|:---:|:---:|:---:|:---:|
| **1.pdf** | 13 | 12 (92%) | 12 (92%) | 7 (54%) |
| **1_original_long.pdf** | 23 | 14 (61%) | 14 (61%) | 1 (4%) |
| **2.pdf** | 15 | 8 (53%) | 8 (53%) | 5 (33%) |
| **4.pdf** | 12 | 12 (100%) | 12 (100%) | 9 (75%) |
| **5.pdf** | 31 | output missing | — | — |
| **6.pdf** | 10 | 5 (50%) | 5 (50%) | 1 (10%) |

In [6]:
display(Markdown('---\n## Detailed Comparison – Per Provision'))

for pair in pairs:
    stem = pair['stem']
    rows = all_results[stem]

    if pair['status'] == 'MISSING_OUTPUT':
        display(Markdown(
            f'### {stem}.pdf\n\n'
            f'> **Output file missing** – `data/output/{stem}.json` khong ton tai.\n'
            f'> Ground truth: `{pair["gt"].name}`'))
        continue
    if pair['status'] != 'OK':
        display(Markdown(f'### {stem}.pdf\n\n> {pair["status"]}'))
        continue

    tl = [f'### {stem}.pdf', '']
    tl.append('| Provision (GT) | GT Keywords | Output Keywords | KW Match | GT Text | Output Exact Text | Text Match |')
    tl.append('|---|---|---|:---:|---|---|:---:|')

    for r in rows:
        prov = r['provision']
        gt_kws_str = ', '.join(f'`{k}`' for k in r['gt_kws'][:5])
        if len(r['gt_kws']) > 5:
            gt_kws_str += f' (+{len(r["gt_kws"])-5})'
        if not r['found']:
            tl.append(f'| {prov} | {gt_kws_str} | NOT FOUND | no | {trunc(r["gt_text"],120)} | — | no |')
            continue
        out_kws_str = ', '.join(f'`{k}`' for k in r['out_kws'][:5])
        if len(r['out_kws']) > 5:
            out_kws_str += f' (+{len(r["out_kws"])-5})'
        kw_cell  = ('yes ' + ', '.join(f'`{h}`' for h in r['kw_hits'][:3])) if r['kw_ok'] else 'no'
        ratio_s  = f"{r['txt_ratio']:.0%}"
        txt_cell = f'yes {ratio_s}' if r['txt_ok'] else f'no {ratio_s}'
        tl.append(f'| {prov} | {gt_kws_str} | {out_kws_str} | {kw_cell} | {trunc(r["gt_text"],140)} | {trunc(r["best_txt"],140)} | {txt_cell} |')

    n = len(rows)
    kw_p  = sum(1 for r in rows if r['kw_ok'])
    txt_p = sum(1 for r in rows if r['txt_ok'])
    kf    = [r['provision'] for r in rows if not r['kw_ok']]
    tf    = [r['provision'] for r in rows if not r['txt_ok']]
    tl.append('')
    tl.append(f'**Keyword**: {kw_p}/{n} passed' +
              (' — failed: ' + ', '.join(f'`{p}`' for p in kf) if kf else ''))
    tl.append(f'**Text**: {txt_p}/{n} passed' +
              (' — failed: ' + ', '.join(f'`{p}`' for p in tf) if tf else ''))
    display(Markdown('\n'.join(tl)))
    display(Markdown('---'))


---
## Detailed Comparison – Per Provision

### 1.pdf

| Provision (GT) | GT Keywords | Output Keywords | KW Match | GT Text | Output Exact Text | Text Match |
|---|---|---|:---:|---|---|:---:|
| Parties / Agreement Intro | `Parties / Agreement Intro`, `Parties`, `Service Provider`, `Client`, `Employer` (+1) | `Parties`, `Orion Digital Solutions Pte. Ltd.`, `Lotus Retail Group Joint Stock Company`, `Service Provider`, `Client` | yes `Parties`, `Service Provider`, `Client` | Short sample contract for keyword grouping and evidence extraction testing This Professional Software Development and Maintenance Agreement ... | This Professional Software Development and Maintenance Agreement (the "Agreement") is made between Orion Digital Solutions Pte. Ltd. ("Servi... | yes 74% |
| Parties | `Parties`, `Service Provider`, `Client`, `Employer`, `Employee` | `Parties`, `Orion Digital Solutions Pte. Ltd.`, `Lotus Retail Group Joint Stock Company`, `Service Provider`, `Client` | yes `Parties`, `Service Provider`, `Client` | 1. Parties The Service Provider is Orion Digital Solutions Pte. Ltd., a Singapore company. The Client is Lotus Retail Group Joint Stock Comp... | This Professional Software Development and Maintenance Agreement (the "Agreement") is made between Orion Digital Solutions Pte. Ltd. ("Servi... | no 34% |
| Effective Date and Contract Term | `Effective Date and Contract Term`, `Effective Date`, `Commencement Date`, `Start Date`, `End Date` (+4) | `Effective Date`, `Commencement Date`, `Start Date`, `End Date`, `Expiration Date` | yes `Effective Date`, `Commencement Date`, `Start Date` | 2. Effective Date and Contract Term Effective Date: April 1, 2026. Commencement Date: April 1, 2026. Start Date: April 1, 2026. End Date: Ma... | Effective Date: April 1, 2026. Commencement Date: April 1, 2026. Start Date: April 1, 2026. End Date: March 31, 2027. Expiration Date: March... | no 41% |
| Scope of Services | `Scope of Services`, `Services`, `Software Development`, `Maintenance` | `Services`, `design`, `develop`, `maintain`, `support` (+8) | yes `Services`, `Software Development` | 3. Scope of Services The Service Provider shall design, develop, maintain, and support a retail operations platform for the Client. The serv... | The Service Provider shall design, develop, maintain, and support a retail operations platform for the Client. The services include requirem... | yes 91% |
| Deliverables and Acceptance | `Deliverables and Acceptance`, `Deliverables`, `Milestones`, `Acceptance Criteria`, `Acceptance` (+2) | NOT FOUND | no | 4. Deliverables and Acceptance The Service Provider shall deliver source code, deployment scripts, user documentation, a... | — | no |
| Payment Terms | `Payment Terms`, `Term`, `Effective Date`, `Expiration`, `Probation` (+3) | `Payment Terms`, `monthly service fee`, `USD 4,500`, `invoice`, `payment due` (+2) | yes `Payment Terms`, `Invoice` | 5. Payment Terms The Client shall pay the Service Provider a monthly service fee of USD 4,500. Each invoice shall be issued on the first bus... | The Client shall pay the Service Provider a monthly service fee of USD 4,500. Each invoice shall be issued on the first business day of each... | yes 96% |
| Renewal Clause | `Renewal Clause`, `Renewal`, `Extension`, `Non-Renewal Notice` | `Renewal`, `automatic renewal`, `successive one-year terms`, `written notice of non-renewal` | yes `Renewal` | 6. Renewal Clause This Agreement shall automatically renew for successive one-year terms unless either party gives written notice of non-ren... | This Agreement shall automatically renew for successive one-year terms unless either party gives written notice of non-renewal at least thir... | yes 57% |
| Termination | `Termination`, `Term`, `Effective Date`, `Expiration`, `Probation` (+2) | `Termination`, `written notice`, `material breach`, `cure the breach` | yes `Termination` | 7. Termination Either party may terminate this Agreement by giving thirty (30) days written notice to the other party. The Client may termin... | Either party may terminate this Agreement by giving thirty (30) days written notice to the other party. The Client may terminate immediately... | yes 95% |
| Confidentiality | `Confidentiality`, `Non-Disclosure`, `Confidential Information` | `Confidentiality`, `non-public business information`, `technical information`, `financial information`, `security information` (+2) | yes `Confidentiality` | 8. Confidentiality Each party shall keep confidential all non-public business, technical, financial, security, and customer information rece... | Each party shall keep confidential all non-public business, technical, financial, security, and customer information received from the other... | yes 95% |
| Data Security and Access Control | `Data Security and Access Control`, `Data Protection`, `Data Security`, `Privacy`, `Access Control` (+1) | `Data Protection`, `administrative safeguards`, `technical safeguards`, `organizational safeguards`, `authorized personnel` (+1) | yes `Data Protection` | 9. Data Security and Access Control The Service Provider shall use reasonable administrative, technical, and organizational safeguards to pr... | The Service Provider shall use reasonable administrative, technical, and organizational safeguards to protect Client data. Access to product... | yes 94% |
| Liability | `Liability`, `Limitation of Liability`, `Damages`, `Liability Cap` | `Liability`, `total liability`, `indirect damages`, `incidental damages`, `special damages` (+2) | yes `Liability` | 10. Liability The Service Provider total liability under this Agreement shall not exceed the total service fees paid by the Client during th... | The Service Provider total liability under this Agreement shall not exceed the total service fees paid by the Client during the three | no 41% |
| Governing Law and Dispute Resolution | `Governing Law and Dispute Resolution`, `Governing Law`, `Dispute Resolution`, `Arbitration`, `Court` | `Governing Law`, `laws of Vietnam`, `dispute resolution`, `good-faith negotiation`, `competent court` | yes `Governing Law`, `Dispute Resolution` | 11. Governing Law and Dispute Resolution This Agreement shall be governed by and interpreted in accordance with the laws of Vietnam. Any dis... | If negotiation fails, the dispute may be submitted to a competent court in Ho Chi Minh City, Vietnam. | no 43% |
| Notices | `Notices`, `Written Notice` | `Notices`, `written notice`, `email`, `registered mail`, `contact address` (+1) | yes `Notices`, `Written Notice` | 12. Notices All notices under this Agreement must be in writing and sent by email or registered mail to the contact address provided by each... | All notices under this Agreement must be in writing and sent by email or registered mail to the contact address provided by each party. | no 44% |

**Keyword**: 12/13 passed — failed: `Deliverables and Acceptance`
**Text**: 7/13 passed — failed: `Parties`, `Effective Date and Contract Term`, `Deliverables and Acceptance`, `Liability`, `Governing Law and Dispute Resolution`, `Notices`

---

### 1_original_long.pdf

| Provision (GT) | GT Keywords | Output Keywords | KW Match | GT Text | Output Exact Text | Text Match |
|---|---|---|:---:|---|---|:---:|
| Parties / Key Contract Fields | `Parties / Key Contract Fields`, `Parties`, `Service Provider`, `Client`, `Employer` (+1) | `Parties`, `Orion Digital Solutions Pte. Ltd.`, `Lotus Retail Group Joint Stock Company`, `Service Provider`, `Client` | yes `Parties`, `Service Provider`, `Client` | PROFESSIONAL SOFTWARE DEVELOPMENT AND MAINTENANCE AGREEMENT Sample contract for keyword grouping, evidence extraction, and exact text valida... | This Professional Software Development and Maintenance Agreement (the "Agreement") is entered into by and between Orion Digital Solutions Pt... | yes 62% |
| Definitions | `Definitions`, `Defined Terms` | NOT FOUND | no | 1. Definitions 1.1 "Deliverables" means the software modules, source code, configuration files, technical documentation,... | — | no |
| Term, Effective Date, and Expiration | `Term, Effective Date, and Expiration`, `Effective Date`, `Commencement Date`, `Start Date`, `End Date` (+4) | `Effective Date`, `April 1, 2026`, `Commencement Date`, `Start Date` | yes `Effective Date`, `Commencement Date`, `Start Date` | 2. Term, Effective Date, and Expiration 2.1 The Effective Date of this Agreement is April 1, 2026. The Commencement Date and Start Date are ... | Effective Date April 1, 2026 Commencement Date April 1, 2026 Start Date April 1, 2026 | no 10% |
| Scope of Services | `Scope of Services`, `Services`, `Software Development`, `Maintenance` | `Services`, `business analysis`, `solution architecture`, `backend API development`, `frontend dashboard development` (+4) | yes `Services` | 3. Scope of Services 3.1 Service Provider shall design, develop, configure, test, deploy, and maintain a cloud-based retail operations platf... | 3.1 Service Provider shall design, develop, configure, test, deploy, and maintain a cloud-based retail operations platform for Client. The s... | no 45% |
| Deliverables, Milestones, and Acceptance | `Deliverables, Milestones, and Acceptance`, `Deliverables`, `Milestones`, `Acceptance Criteria`, `Acceptance` (+2) | `Milestones`, `Discovery Completion`, `Prototype Delivery`, `User Acceptance Testing Release`, `Production Go-Live` | yes `Milestones` | 4. Deliverables, Milestones, and Acceptance 4.1 Service Provider shall deliver the Deliverables according to the project plan and milestone ... | The initial milestone schedule includes: Discovery Completion by April 15, 2026; Prototype Delivery by May 15, 2026; User Acceptance Testing... | no 28% |
| Fees, Payment Terms, Invoices, and Taxes | `Fees, Payment Terms, Invoices, and Taxes`, `Term`, `Effective Date`, `Expiration`, `Probation` (+7) | `Term`, `Twelve (12) months`, `Initial Term`, `Expiration Date`, `March 31, 2027` | yes `Term` | 5. Fees, Payment Terms, Invoices, and Taxes 5.1 Client shall pay Service Provider a fixed implementation fee of USD 48,000 and a monthly mai... |  | no 0% |
| Renewal, Extension, and Non-Renewal Notice | `Renewal, Extension, and Non-Renewal Notice`, `Renewal`, `Extension`, `Non-Renewal Notice`, `Notices` (+1) | `Extension`, `approved in writing`, `both Parties` | yes `Extension` | 6. Renewal, Extension, and Non-Renewal Notice 6.1 This Agreement shall automatically renew for successive one-year renewal terms unless eith... | 6.3 Either Party may propose an extension of scope or term by submitting a written Change Request. An extension is not effective unless appr... | no 38% |
| Termination and Suspension | `Termination and Suspension`, `Term`, `Effective Date`, `Expiration`, `Probation` (+3) | `Term`, `Twelve (12) months`, `Initial Term`, `Expiration Date`, `March 31, 2027` | yes `Term` | 7. Termination and Suspension 7.1 Either Party may terminate this Agreement for convenience by giving sixty (60) days written notice to the ... |  | no 0% |
| Service Levels, Support, and Maintenance | `Service Levels, Support, and Maintenance` | NOT FOUND | no | 8. Service Levels, Support, and Maintenance 8.1 Service Provider shall provide support from Monday to Friday, 9:00 a.m. ... | — | no |
| Change Requests | `Change Requests` | NOT FOUND | no | 9. Change Requests 9.1 Any requested change to scope, timeline, fees, Deliverables, Acceptance Criteria, integration req... | — | no |
| Confidentiality and Non-Disclosure | `Confidentiality and Non-Disclosure`, `Confidentiality`, `Non-Disclosure`, `Confidential Information` | NOT FOUND | no | 10. Confidentiality and Non-Disclosure 10.1 Each Party shall protect the other Party’s Confidential Information using at... | — | no |
| Data Protection and Security | `Data Protection and Security`, `Data Protection`, `Data Security`, `Privacy`, `Access Control` (+1) | NOT FOUND | no | 11. Data Protection and Security 11.1 Service Provider shall implement reasonable administrative, technical, and organiz... | — | no |
| Intellectual Property Rights and License | `Intellectual Property Rights and License`, `Intellectual Property`, `License`, `Source Code`, `Ownership` (+3) | NOT FOUND | no | 12. Intellectual Property Rights and License 12.1 Subject to full payment of all undisputed fees, Client will own the cu... | — | no |
| Warranties and Disclaimers | `Warranties and Disclaimers`, `Warranties`, `Disclaimers` | `Warranties`, `Disclaimers` | yes `Warranties`, `Disclaimers` | 13. Warranties and Disclaimers 13.1 Service Provider warrants that the services will be performed in a professional and workmanlike manner b... | 13.1 Service Provider warrants that the services will be performed in a professional and workmanlike manner by personnel with appropriate sk... | no 33% |
| Indemnification | `Indemnification`, `Third-Party Claims` | `Indemnification` | yes `Indemnification` | 14. Indemnification 14.1 Service Provider shall defend, indemnify, and hold harmless Client against third-party claims alleging that the cus... | 14.1 Service Provider shall defend, indemnify, and hold harmless Client against third-party claims alleging that the custom Deliverables inf... | no 54% |
| Limitation of Liability | `Limitation of Liability`, `Damages`, `Liability Cap` | `Limitation of Liability` | yes `Limitation of Liability` | 15. Limitation of Liability 15.1 The total liability of either Party arising out of or related to this Agreement shall not exceed the total ... | 15.1 The total liability of either Party arising out of or related to this Agreement shall not exceed the total fees paid or payable by Clie... | no 41% |
| Audit Rights and Records | `Audit Rights and Records`, `Audit Rights`, `Records` | `Audit Rights`, `Records` | yes `Audit Rights`, `Records` | 16. Audit Rights and Records 16.1 During the term and for two (2) years after termination or expiration, each Party shall maintain accurate ... | 16.1 During the term and for two (2) years after termination or expiration, each Party shall maintain accurate records reasonably necessary ... | no 45% |
| Subcontracting and Assignment | `Subcontracting and Assignment`, `Subcontracting`, `Assignment`, `Non-assignment` | NOT FOUND | no | 17. Subcontracting and Assignment 17.1 Service Provider may use qualified subcontractors to perform portions of the serv... | — | no |
| Force Majeure | `Force Majeure` | NOT FOUND | no | 18. Force Majeure 18.1 Neither Party will be liable for delay or failure to perform caused by events beyond its reasonab... | — | no |
| Governing Law and Dispute Resolution | `Governing Law and Dispute Resolution`, `Governing Law`, `Dispute Resolution`, `Arbitration`, `Court` | `Governing Law`, `Laws of Vietnam` | yes `Governing Law` | 19. Governing Law and Dispute Resolution 19.1 This Agreement shall be governed by and interpreted in accordance with the laws of Vietnam, wi... | Governing Law Laws of Vietnam | no 7% |
| Notices | `Notices`, `Written Notice` | `Notices`, `legal@oriondigital.sg`, `delivery@oriondigital.sg` | yes `Notices` | 20. Notices 20.1 All notices under this Agreement must be in writing and delivered by email with confirmation of receipt, registered mail, c... | 20.2 Notices to Client must be sent to legal@lotusretail.vn with a copy to procurement@lotusretail.vn. Notices to Service Provider must be s... | no 36% |
| Entire Agreement, Order of Precedence, and Amendments | `Entire Agreement, Order of Precedence, and Amendments`, `Entire Agreement`, `Amendments`, `Order of Precedence` | NOT FOUND | no | 21. Entire Agreement, Order of Precedence, and Amendments 21.1 This Agreement, together with any signed Statement of Wor... | — | no |
| Signatures | `Signatures`, `Execution` | `Signatures`, `Authorized Representative`, `Name`, `Title` | yes `Signatures` | 22. Signatures By signing below, the Parties confirm that they have read, understood, and agreed to be bound by this Agreement. Service Prov... | By signing below, the Parties confirm that they have read, understood, and agreed to be bound by this Agreement. | no 44% |

**Keyword**: 14/23 passed — failed: `Definitions`, `Service Levels, Support, and Maintenance`, `Change Requests`, `Confidentiality and Non-Disclosure`, `Data Protection and Security`, `Intellectual Property Rights and License`, `Subcontracting and Assignment`, `Force Majeure`, `Entire Agreement, Order of Precedence, and Amendments`
**Text**: 1/23 passed — failed: `Definitions`, `Term, Effective Date, and Expiration`, `Scope of Services`, `Deliverables, Milestones, and Acceptance`, `Fees, Payment Terms, Invoices, and Taxes`, `Renewal, Extension, and Non-Renewal Notice`, `Termination and Suspension`, `Service Levels, Support, and Maintenance`, `Change Requests`, `Confidentiality and Non-Disclosure`, `Data Protection and Security`, `Intellectual Property Rights and License`, `Warranties and Disclaimers`, `Indemnification`, `Limitation of Liability`, `Audit Rights and Records`, `Subcontracting and Assignment`, `Force Majeure`, `Governing Law and Dispute Resolution`, `Notices`, `Entire Agreement, Order of Precedence, and Amendments`, `Signatures`

---

...

### 4.pdf

| Provision (GT) | GT Keywords | Output Keywords | KW Match | GT Text | Output Exact Text | Text Match |
|---|---|---|:---:|---|---|:---:|
| Parties / Agreement Intro | `Parties / Agreement Intro`, `Parties`, `Service Provider`, `Client`, `Employer` (+1) | `Parties`, `Alpha Solutions Ltd.`, `Beta Retail JSC`, `Service Provider`, `Client` | yes `Parties`, `Service Provider`, `Client` | This Basic Service Agreement (the "Agreement") is made between Alpha Solutions Ltd. ("Service Provider") and Beta Retail JSC ("Client"). | This Basic Service Agreement (the "Agreement") is made between Alpha Solutions Ltd. ("Service Provider") and Beta Retail JSC ("Client"). | yes 100% |
| Parties | `Parties`, `Service Provider`, `Client`, `Employer`, `Employee` | `Parties`, `Alpha Solutions Ltd.`, `Beta Retail JSC`, `Service Provider`, `Client` | yes `Parties`, `Service Provider`, `Client` | 1. Parties Alpha Solutions Ltd. will provide software support and maintenance services to Beta Retail JSC. The parties agree to follow the t... | This Basic Service Agreement (the "Agreement") is made between Alpha Solutions Ltd. ("Service Provider") and Beta Retail JSC ("Client"). | no 28% |
| Effective Date and Contract Term | `Effective Date and Contract Term`, `Effective Date`, `Commencement Date`, `Start Date`, `End Date` (+4) | `Effective Date`, `Commencement Date`, `Start Date`, `End Date`, `Expiration Date` | yes `Effective Date`, `Commencement Date`, `Start Date` | 2. Effective Date and Contract Term Effective Date: January 1, 2026. Commencement Date: January 1, 2026. Start Date: January 1, 2026. End Da... | Effective Date: January 1, 2026. Commencement Date: January 1, 2026. Start Date: January 1, 2026. End Date: December 31, 2026. Expiration Da... | yes 87% |
| Scope of Services | `Scope of Services`, `Services`, `Software Development`, `Maintenance` | `Services`, `software support`, `maintenance services`, `system monitoring`, `bug fixing` (+2) | yes `Services` | 3. Scope of Services The Service Provider shall provide system monitoring, bug fixing, monthly maintenance, and technical support for the Cl... | Alpha Solutions Ltd. will provide software support and maintenance services to Beta Retail JSC. | no 19% |
| Payment Terms | `Payment Terms`, `Term`, `Effective Date`, `Expiration`, `Probation` (+3) | `Payment Terms`, `monthly service fee`, `USD 2,000`, `invoice`, `payment due` | yes `Payment Terms`, `Invoice` | 4. Payment Terms The Client shall pay the Service Provider a monthly service fee of USD 2,000. Each invoice shall be issued on the first bus... | The Client shall pay the Service Provider a monthly service fee of USD 2,000. Each invoice shall be issued on the first business day of each... | yes 94% |
| Renewal Clause | `Renewal Clause`, `Renewal`, `Extension`, `Non-Renewal Notice` | `Renewal`, `automatic renewal`, `successive one-year terms`, `written notice` | yes `Renewal` | 5. Renewal Clause This Agreement shall automatically renew for successive one-year terms unless either party gives written notice of non-ren... | This Agreement shall automatically renew for successive one-year terms unless either party gives written notice of non-renewal at least thir... | yes 95% |
| Termination | `Termination`, `Term`, `Effective Date`, `Expiration`, `Probation` (+2) | `Termination`, `written notice`, `material breach`, `cure the breach` | yes `Termination` | 6. Termination Either party may terminate this Agreement by giving thirty (30) days' written notice to the other party. The Client may termi... | Either party may terminate this Agreement by giving thirty (30) days' written notice to the other party. The Client may terminate immediatel... | yes 95% |
| Confidentiality | `Confidentiality`, `Non-Disclosure`, `Confidential Information` | `Confidentiality`, `non-public business`, `technical information`, `financial information`, `customer information` | yes `Confidentiality` | 7. Confidentiality Each party shall keep confidential all non-public business, technical, financial, and customer information received from ... | Each party shall keep confidential all non-public business, technical, financial, and customer information received from the other party. | no 50% |
| Liability | `Liability`, `Limitation of Liability`, `Damages`, `Liability Cap` | `Liability`, `total liability`, `service fees`, `indirect damages`, `incidental damages` (+1) | yes `Liability` | 8. Liability The Service Provider's total liability under this Agreement shall not exceed the total service fees paid by the Client during t... | The Service Provider's total liability under this Agreement shall not exceed the total service fees paid by the Client during the three (3) ... | yes 97% |
| Governing Law | `Governing Law`, `Dispute Resolution`, `Arbitration`, `Court` | `Governing Law`, `laws of Vietnam`, `good-faith negotiation` | yes `Governing Law` | 9. Governing Law This Agreement shall be governed by and interpreted in accordance with the laws of Vietnam. Any dispute arising from this A... | This Agreement shall be governed by and interpreted in accordance with the laws of Vietnam. Any dispute arising from this Agreement shall fi... | yes 90% |
| Notices | `Notices`, `Written Notice` | `Notices`, `in writing`, `email`, `registered mail` | yes `Notices` | 10. Notices All notices under this Agreement must be in writing and sent by email or registered mail to the contact address provided by each... | All notices under this Agreement must be in writing and sent by email or registered mail to the contact address provided by each party. | yes 96% |
| Signatures | `Signatures`, `Execution` | `Signatures`, `Authorized Representative`, `Date` | yes `Signatures` | 11. Signatures By signing below, the parties agree to be bound by this Agreement. Service Provider: Alpha Solutions Ltd. Authorized Represen... | 11. Signatures By signing below, the parties agree to be bound by this Agreement. Service Provider: Alpha Solutions Ltd. Authorized Represen... | yes 100% |

**Keyword**: 12/12 passed
**Text**: 9/12 passed — failed: `Parties`, `Scope of Services`, `Confidentiality`

---

### 5.pdf

> **Output file missing** – `data/output/5.json` khong ton tai.
> Ground truth: `ground_truth_5.json`

### 6.pdf

| Provision (GT) | GT Keywords | Output Keywords | KW Match | GT Text | Output Exact Text | Text Match |
|---|---|---|:---:|---|---|:---:|
| Parties / Recitals | `Parties / Recitals`, `Parties`, `Service Provider`, `Client`, `Employer` (+1) | `Parties`, `CNS Pharmaceuticals, Inc.`, `WPD Pharmaceuticals`, `Party`, `Parties` | yes `Parties` | This Development Agreement (the “Agreement”) dated as of March 20, 2020 (the “Effective Date”) is entered into by and between CNS Pharmaceut... | CNS and WPD are sometimes referred to herein individually as a “ Party” and collectively as the “ Parties.” | no 11% |
| Definitions | `Definitions`, `Defined Terms` | NOT FOUND | no | ARTICLE 1 DEFINITIONS 1.1 “Approval Achievement Date” means the earlier of the: (i) date on which WPD receives marketing... | — | no |
| Development Agreement | `Development Agreement`, `Commercially Reasonable Efforts`, `Development Products`, `Development Fees`, `Milestone Payment` | `Development Fees`, `payment by WPD to CNS`, `Development Fees`, `$1.0 million` | yes `Development Fees` | ARTICLE 2 DEVELOPMENT AGREEMENT 2.1 Subject to the terms and conditions of this Agreement, WPD hereby agrees to use its commercially reasona... | or (ii) the payment by WPD to CNS of Development Fees hereunder of $1.0 million. | no 7% |
| Information And Use | `Information And Use`, `Information`, `Use`, `Reports`, `Know-How` | NOT FOUND | no | ARTICLE 3 INFORMATION AND USE 3.1 WPD shall furnish CNS with written reports summarizing the progress of the research an... | — | no |
| Other Compensation | `Other Compensation`, `Buyback Consideration` | `Buyback Consideration`, `50% of the Buyback Consideration`, `amounts actually provided` | yes `Buyback Consideration` | ARTICLE 4 OTHER COMPENSATION 4.1 If MBI exercises its right to terminate the Sublicense Agreement in whole, or to remove a portion of the su... | WPD agrees that CNS shall receive the greater of (i) 50% of the Buyback Consideration that is attributable to the field of anti-viral pharma... | yes 57% |
| Confidentiality | `Confidentiality`, `Non-Disclosure`, `Confidential Information` | `Confidential Information`, `confidential`, `disclosing party`, `recipient party`, `proprietary technical information` (+1) | yes `Confidential Information` | ARTICLE 5 CONFIDENTIALITY 5.1 During the term of this Agreement and for a period of five (5) years thereafter, the Parties each agree that C... | 1.4 “Confidential Information” includes: (1) all information contained in documents marked "confidential" and disclosed by one Party (the "d... | no 4% |
| Term And Termination | `Term And Termination`, `Term`, `Effective Date`, `Expiration`, `Probation` (+3) | `Effective Date`, `Effective Date`, `March 20, 2020` | yes `Effective Date` | ARTICLE 6 TERM AND TERMINATION 6.1 The term of this Agreement will commence on the Effective Date and remain in full force and effect until ... | This Development Agreement (the “ Agreement”) dated as of March 20, 2020 (the “ Effective Date”) is entered into by and between CNS Pharmace... | no 9% |
| Representations, Warranties And Covenants | `Representations, Warranties And Covenants`, `Warranties`, `Disclaimers`, `Representations`, `Covenants` | NOT FOUND | no | ARTICLE 7 REPRESENTATIONS, WARRANTIES AND COVENANTS 7.1 Each Party represents and warrants that: 7.1.1 it is duly organi... | — | no |
| Indemnification | `Indemnification`, `Third-Party Claims` | NOT FOUND | no | ARTICLE 8 INDEMNIFICATION 8.1 WPD hereby agrees to hold harmless and indemnify CNS, its officers, affiliates, employees,... | — | no |
| Miscellaneous | `Miscellaneous`, `Further Assurances`, `Counterparts`, `Severability` | NOT FOUND | no | ARTICLE 9 MISCELLANEOUS 9.1 The Parties shall execute and deliver any and all additional papers, documents, and other in... | — | no |

**Keyword**: 5/10 passed — failed: `Definitions`, `Information And Use`, `Representations, Warranties And Covenants`, `Indemnification`, `Miscellaneous`
**Text**: 1/10 passed — failed: `Parties / Recitals`, `Definitions`, `Development Agreement`, `Information And Use`, `Confidentiality`, `Term And Termination`, `Representations, Warranties And Covenants`, `Indemnification`, `Miscellaneous`

---